# BBO Capstone Method Summary

This notebook is a lightweight walkthrough of the optimisation approach used in the Black-Box Optimisation capstone project.

It is not intended to reproduce the exact weekly portal results, because the original weekly query outputs and intermediate `.npy` datasets are not included in this public repository. Instead, it documents the method, shows the final visualisations, and runs a small synthetic example that demonstrates the Gaussian Process and acquisition-function logic used in the project.


## Repository context

The main implementation lives in:

`capstone/src/`

The final reports and documentation live in:

`capstone/reports/`  
`capstone/docs/`

The final visualisations live in:

`capstone/data/`


In [ ]:
from pathlib import Path
import sys

projectRoot = Path.cwd()

if not (projectRoot / "capstone").exists():
    projectRoot = projectRoot.parent

srcPath = projectRoot / "capstone" / "src"
dataPath = projectRoot / "capstone" / "data"

sys.path.append(str(srcPath))

print("Project root:", projectRoot)
print("Source path exists:", srcPath.exists())
print("Data path exists:", dataPath.exists())


## Final optimisation progress

The following charts summarise the final optimisation trajectory recorded in the repository.


In [ ]:
from IPython.display import Image, display

perFunctionChart = dataPath / "bbo_progress_per_function.png"
normalisedChart = dataPath / "bbo_progress_normalised.png"

if perFunctionChart.exists():
    display(Image(filename=str(perFunctionChart)))
else:
    print("Missing chart:", perFunctionChart)

if normalisedChart.exists():
    display(Image(filename=str(normalisedChart)))
else:
    print("Missing chart:", normalisedChart)


## Synthetic demonstration

The real course data is not included, so this section uses a tiny synthetic one-dimensional optimisation example. The purpose is to show that the core GP posterior and acquisition logic can be imported and executed from the project code.

This does not represent any of the hidden capstone functions.


In [ ]:
import numpy as np
from suggest_queries_week_13_gp import gpPosterior, normalCdf, normalPdf

np.random.seed(42)

xTrain = np.array([[0.10], [0.25], [0.45], [0.70], [0.90]], dtype=np.float64)
yTrain = np.sin(8.0 * xTrain).reshape(-1) + 0.5 * xTrain.reshape(-1)

xCandidates = np.linspace(0.0, 0.999999, 250).reshape(-1, 1)

yMeanValue = float(np.mean(yTrain))
yStdValue = float(np.std(yTrain))
yTrainNormalised = (yTrain - yMeanValue) / yStdValue

meanNormalised, varianceNormalised = gpPosterior(
    xTrain=xTrain,
    yTrain=yTrainNormalised,
    xTest=xCandidates,
    lengthScale=0.2,
    noiseLevel=1e-8
)

sigmaNormalised = np.sqrt(varianceNormalised)
bestObservedNormalised = float(np.max(yTrainNormalised))

xi = 1e-6
improvement = meanNormalised - bestObservedNormalised - xi
zScore = improvement / (sigmaNormalised + 1e-12)
expectedImprovement = improvement * normalCdf(zScore) + sigmaNormalised * normalPdf(zScore)
expectedImprovement = np.maximum(expectedImprovement, 0.0)

bestIndex = int(np.argmax(expectedImprovement))
nextSuggestion = xCandidates[bestIndex]

print("Synthetic training inputs:", xTrain.reshape(-1))
print("Synthetic training outputs:", np.round(yTrain, 4))
print("Suggested next point:", f"{float(nextSuggestion[0]):.6f}")
print("Expected improvement score:", float(expectedImprovement[bestIndex]))


## Interpretation

The synthetic run demonstrates the same core pattern used in the capstone project:

1. Fit a Gaussian Process surrogate to limited observations.
2. Estimate uncertainty across candidate points.
3. Use an acquisition function such as Expected Improvement to choose the next query.
4. Adapt the strategy as more feedback becomes available.

In the actual capstone work, this process was repeated function by function and week by week, with manual strategy changes documented in the weekly reports.


## Reproducibility note

The repository contains the implementation, reports, model card, datasheet, and final visualisations. Exact numerical reproduction of the weekly submissions requires the original query results from the course platform, which are not included in this public repository.
